<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 15. GCN — Semi-Supervised Classification with Graph Convolutional Networks

- **원 논문:** [Semi-Supervised Classification with Graph Convolutional Networks](https://arxiv.org/abs/1609.02907)
- **저자 / 발표:** Thomas N. Kipf and Max Welling · ICLR (2017)
- **난이도 / 예상 시간:** 중급 · 약 55분
- **선수 지식:** PyTorch 기본, 행렬곱, 그래프 adjacency/degree, cross entropy

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

두 community의 작은 합성 그래프에서 self-loop와 대칭 degree normalization을 직접 구현하고, 노드 네 개의 label만으로 2-layer GCN이 나머지 노드를 분류하는지 확인한다.

**원 논문과 다른 것**

논문은 Cora, Citeseer, Pubmed 등의 citation graph에서 sparse 연산을 사용한다. 여기서는 24-node dense adjacency를 사용하지만 Eq. (2), Eq. (8)–(10)의 전파와 masked loss는 같다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2, Eq. (2) | `GraphConvolution.forward`의 `A_hat @ X @ W` 전파 | node 수를 보존하는 layer 출력 shape와 finite gradient를 검사 |
| §2.2, Eq. (7)–(8) | `normalize_adjacency`의 self-loop와 대칭 degree normalization | 정규화 행렬의 shape·대칭성·유한값을 검사 |
| §3.1, Eq. (9)–(10) | `TinyGCN`의 2-layer propagation과 train mask cross entropy | 미공개 test node 정확도와 학습 loss 감소를 검사 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
H^{(l+1)}=\sigma\!\left(
\hat D^{-1/2}\hat A\hat D^{-1/2}H^{(l)}W^{(l)}\right),
\qquad \hat A=A+I
$$

- $\hat A$는 self-loop를 추가한 adjacency, $\hat D$는 degree matrix입니다.
- symmetric normalization은 이웃 수가 다른 node의 message 크기를 보정합니다.
- adjacency 준비, normalization, graph convolution layer, node loss를 직접 분리합니다.
- $H^{(l)}:[N,D_l]$는 normalized adjacency $[N,N]$와 가중치 $[D_l,D_{l+1}]$를 거칩니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2, Eq. (2)
- **노트북 구현:** `GraphConvolution.forward`의 `A_hat @ X @ W` 전파
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** node 수를 보존하는 layer 출력 shape와 finite gradient를 검사를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 논문은 Cora, Citeseer, Pubmed 등의 citation graph에서 sparse 연산을 사용한다. 여기서는 24-node dense adjacency를 사용하지만 Eq. (2), Eq. (8)–(10)의 전파와 masked loss는 같다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. Citation graph를 두 community로 축소하기

논문 §2 Eq. (2)의 층은

$$H^{(l+1)}=\sigma(\tilde D^{-1/2}\tilde A\tilde D^{-1/2}H^{(l)}W^{(l)})$$

입니다. §2.2 Eq. (8)의 `renormalization trick`은 $\tilde A=A+I$를 사용합니다.
여기서는 두 community를 가진 24-node graph와 약한 noisy feature를 만들고, 각 class에서
단 두 노드만 label로 공개합니다. 그래프 이웃 집계가 약한 feature 신호를 매끄럽게 만듭니다.

**성공 기준:** 공개하지 않은 20개 노드의 정확도가 85%보다 커야 합니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** A [N,N], X [N,D] → graph hidden [N,H] → node logits [N,K]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(150)
np.random.seed(150)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# This 24-node dense example may be faster with AI_LAB_DEVICE=cpu;
# auto acceleration preserves the same code path for larger graphs.

n_per_class = 12
n_nodes = 2 * n_per_class
labels = torch.tensor([0] * n_per_class + [1] * n_per_class, dtype=torch.long)
adjacency = torch.zeros(n_nodes, n_nodes)

# 각 community 안에서 ring의 1-hop/2-hop 이웃을 연결한다.
for offset in (0, n_per_class):
    for i in range(n_per_class):
        for delta in (1, 2):
            j = offset + (i + delta) % n_per_class
            k = offset + i
            adjacency[k, j] = adjacency[j, k] = 1.0
# 소수의 community 간 edge는 실제 그래프의 noise 역할을 한다.
for left, right in ((2, 14), (8, 20)):
    adjacency[left, right] = adjacency[right, left] = 1.0

feature_rng = torch.Generator().manual_seed(151)
weak_signal = (2 * labels.float() - 1.0)[:, None] * 0.45
features = torch.cat(
    [
        weak_signal + 0.65 * torch.randn(n_nodes, 1, generator=feature_rng),
        0.75 * torch.randn(n_nodes, 2, generator=feature_rng),
        torch.ones(n_nodes, 1),
    ],
    dim=1,
)
train_idx = torch.tensor([0, 5, 12, 17])
test_mask = torch.ones(n_nodes, dtype=torch.bool)
test_mask[train_idx] = False
labels, adjacency, features, train_idx, test_mask = ACCELERATOR.move(
    labels, adjacency, features, train_idx, test_mask
)

assert torch.allclose(adjacency, adjacency.T)
assert features.shape == (24, 4)
print(ACCELERATOR.summary())
edge_count = int(adjacency.sum().item() / 2)
print(f"nodes={n_nodes}, undirected edges={edge_count}, labels used=4")

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2.2, Eq. (7)–(8)
- **이 셀의 책임:** `normalize_adjacency`의 self-loop와 대칭 degree normalization
- **Tensor shape 계약:** A [N,N], X [N,D] → graph hidden [N,H] → node logits [N,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 정규화 행렬의 shape·대칭성·유한값을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: A_tilde=A+I와 D_tilde^(-1/2) A_tilde D_tilde^(-1/2)를 구현하세요.
def normalize_adjacency(adjacency):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


a_hat = normalize_adjacency(adjacency)
assert a_hat.shape == adjacency.shape
assert torch.allclose(a_hat, a_hat.T, atol=1e-06)
assert torch.isfinite(a_hat).all()


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2, Eq. (2) / §3.1, Eq. (9)–(10)
- **이 셀의 책임:** `GraphConvolution.forward`의 `A_hat @ X @ W` 전파 / `TinyGCN`의 2-layer propagation과 train mask cross entropy
- **Tensor shape 계약:** A [N,N], X [N,D] → graph hidden [N,H] → node logits [N,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** node 수를 보존하는 layer 출력 shape와 finite gradient를 검사 / 미공개 test node 정확도와 학습 loss 감소를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: 논문 식의 W만 정의하세요(bias 없음).
# TODO: A_hat @ X @ W를 반환하세요.
# TODO: 4 -> 12 -> 2 두 층을 정의하세요.
class GraphConvolution(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, in_features, out_features):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x, normalized_adjacency):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


class TinyGCN(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x, normalized_adjacency):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


gcn = TinyGCN().to(DEVICE)
logits = gcn(features, a_hat)
assert logits.shape == (n_nodes, 2)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §3.1, Eq. (9)–(10)
- **이 셀의 책임:** `TinyGCN`의 2-layer propagation과 train mask cross entropy
- **Tensor shape 계약:** A [N,N], X [N,D] → graph hidden [N,H] → node logits [N,K]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 미공개 test node 정확도와 학습 loss 감소를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: Eq. (10)처럼 train_idx의 네 label에 대해서만 cross entropy를 계산해 250회 학습하세요.
optimizer = torch.optim.Adam(gcn.parameters(), lr=0.03, weight_decay=5e-4)
loss_history = []
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §3.1, Eq. (9)–(10)
- **이 셀의 책임:** `TinyGCN`의 2-layer propagation과 train mask cross entropy
- **Tensor shape 계약:** A [N,N], X [N,D] → graph hidden [N,H] → node logits [N,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 미공개 test node 정확도와 학습 loss 감소를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: train/test accuracy를 계산하고 true/predicted graph 및 loss를 그리세요.
def draw_graph(axis, colors, title):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 7. 해석 TODO

1. self-loop를 제거하면 한 층에서 자기 feature는 어떻게 되나요?
2. loss에 `test_mask`를 포함하면 왜 semi-supervised 실험이 아니게 되나요?
3. 이 dense 구현과 논문의 citation graph 구현 사이 계산 복잡도 차이를 쓰세요.

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.